# r2x spike S3: compare step
Throwaway. Runs after `totals` and `counts` (see `after` in `process.yml`). Checks both wrote the same regions for this request; it fails if either step had not finished first.

In [0]:
%pip install -q /Volumes/sconset-db-01/r2x_spike/libraries/r2x_helper-0.1.0-py3-none-any.whl pyyaml==6.0.3

In [0]:
import r2x_helper
v = r2x_helper.get_values()
print(v)

In [0]:
import json
q = "SELECT region FROM IDENTIFIER(:t) WHERE request_id = :rid"
t_regions = {r[0] for r in spark.sql(q, args={"t": v.outputs["totals"], "rid": v.request_id}).collect()}
c_regions = {r[0] for r in spark.sql(q, args={"t": v.outputs["counts"], "rid": v.request_id}).collect()}
if not t_regions or t_regions != c_regions:
    raise RuntimeError(f"compare: totals {sorted(t_regions)} vs counts {sorted(c_regions)} for {v.request_id}")
out = {"step": "compare", "source": v.source, "request_id": v.request_id, "regions": sorted(t_regions)}
print(json.dumps(out))
dbutils.notebook.exit(json.dumps(out))